# Task 3 - Aprendizaje Supervisado

Este notebook funciona como registro incremental de experimentos supervisados para predecir `condition`. La metrica principal sera `recall_condition_1`, porque en un escenario medico interesa reducir falsos negativos: pacientes con condicion real que el modelo clasifica como sanos.

## Estrategia de validacion

- Dataset base: `data/processed/foot_clean.csv`.
- Test final estratificado: 20% reservado hasta el cierre del experimento.
- Validacion interna: `RepeatedStratifiedKFold`, 5 folds y 30 repeticiones.
- Preprocesamiento dentro de `Pipeline`: estandarizacion de continuas, one-hot encoding de discretas y paso directo de binarias.
- Coste medico inicial: `FN=5`, `FP=1`, normalizado por paciente.

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

PROJECT_ROOT

WindowsPath('e:/University/4to/ML/mundial_project')

## Experimento 1 - Regresion Logistica

La regresion logistica se usa como primera linea base porque entrega probabilidades, es interpretable y permite contrastar rapidamente si una frontera lineal captura senales utiles para `condition`.

In [2]:
from task3_logistic_regression import run

result = run()
result["output_dir"]

c:\Users\Josue\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\_plotting.py:175: FutureWarning: `**kwargs` is deprecated and will be removed in 1.9. Pass all matplotlib arguments to `curve_kwargs` as a dictionary instead.
  warnings.warn(


CSV usado: E:\University\4to\ML\mundial_project\data\processed\foot_clean.csv
Resultados guardados en: E:\University\4to\ML\mundial_project\outputs\task3\logistic_regression
               metric     mean      std      min      max
             accuracy 0.849770 0.050816 0.723404 0.958333
precision_condition_1 0.857264 0.065266 0.680000 1.000000
   recall_condition_1 0.813521 0.085339 0.590909 1.000000
       f1_condition_1 0.831736 0.059171 0.684211 0.954545
              roc_auc 0.915978 0.035063 0.789091 0.989091
         medical_cost 0.493505 0.198389 0.085106 1.021277
      false_negatives 4.066667 1.866938 0.000000 9.000000
      false_positives 3.053333 1.633486 0.000000 8.000000
Metricas de test final:
 accuracy  precision_condition_1  recall_condition_1  f1_condition_1  medical_cost  roc_auc  true_negatives  false_positives  false_negatives  true_positives
 0.816667               0.793103            0.821429        0.807018      0.516667 0.896205            26.0              6

WindowsPath('E:/University/4to/ML/mundial_project/outputs/task3/logistic_regression')

In [3]:
cv_summary = pd.read_csv(result["output_dir"] / "cv_metricas_resumen.csv")
cv_summary

,metric,mean,std,min,max
0,accuracy,0.849770,0.050816,0.723404,0.958333
1,precision_condition_1,0.857264,0.065266,0.680000,1.000000
2,recall_condition_1,0.813521,0.085339,0.590909,1.000000
3,f1_condition_1,0.831736,0.059171,0.684211,0.954545
4,roc_auc,0.915978,0.035063,0.789091,0.989091
5,medical_cost,0.493505,0.198389,0.085106,1.021277
6,false_negatives,4.066667,1.866938,0.000000,9.000000
7,false_positives,3.053333,1.633486,0.000000,8.000000


In [4]:
test_metrics = pd.read_csv(result["output_dir"] / "test_metricas.csv")
test_metrics

,accuracy,precision_condition_1,recall_condition_1,f1_condition_1,medical_cost,roc_auc,true_negatives,false_positives,false_negatives,true_positives
0,0.816667,0.793103,0.821429,0.807018,0.516667,0.896205,26.0,6.0,5.0,23.0


## Lectura medica inicial

Para decidir si el modelo es aceptable, revisar primero `recall_condition_1`, falsos negativos y coste medico. La accuracy queda como metrica secundaria, porque puede ocultar errores clinicamente mas graves.